# 03 - Benchmark Reproducible de Metaheurísticas

Experimento principal del artículo. Compara GA, PSO, ABC y TPE para optimización
de hiperparámetros de MLP, LSTM y GRU sobre el dataset PVGIS-ERA5.

**Diseño:** 3 modelos × 4 optimizadores × 30 seeds = **360 experimentos**  
**Budget:** 100 evaluaciones por run  
**Tracking:** MLflow (experimento `solar_benchmark_pvgis`)  

Para monitorear mientras corre:
```bash
mlflow ui --port 5000
# Abrir http://localhost:5000
```

## 1. Imports y configuración

In [ ]:
import os, json, time, random, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import joblib
import mlflow
import optuna
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, LSTM, GRU, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Permitir crecimiento de memoria GPU
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('MLflow:', mlflow.__version__)

In [ ]:
# ── Directorios ────────────────────────────────────────────────────────────────
BASE_DIR    = '/home/sergioperez/servidorjs'
PVGIS_DIR   = os.path.join(BASE_DIR, 'outputs', 'pvgis')
ARRAYS_DIR  = os.path.join(PVGIS_DIR, 'arrays')
SCALERS_DIR = os.path.join(PVGIS_DIR, 'scalers')
OPT_DIR     = os.path.join(PVGIS_DIR, 'benchmark')
MODELS_DIR  = os.path.join(OPT_DIR, 'models')
RESULTS_DIR = os.path.join(OPT_DIR, 'results')

for d in [OPT_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Parámetros del benchmark ───────────────────────────────────────────────────
N_RUNS          = 30      # seeds independientes
BUDGET          = 100     # evaluaciones del objetivo por run
OPT_EPOCHS      = 10      # epochs max durante optimización
OPT_PATIENCE    = 3       # early stopping durante optimización
FINAL_EPOCHS    = 100     # epochs max en evaluación final
FINAL_PATIENCE  = 10
BATCH_CHOICES   = [32, 64, 128]

MODEL_NAMES     = ['MLP', 'LSTM', 'GRU']
OPTIMIZER_NAMES = ['PSO', 'GA', 'ABC', 'TPE']

# ── MLflow ─────────────────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT = 'solar_benchmark_pvgis'
MLFLOW_TRACKING   = os.path.join(BASE_DIR, 'mlruns')
mlflow.set_tracking_uri(f'file://{MLFLOW_TRACKING}')
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f'Experimento MLflow: {MLFLOW_EXPERIMENT}')
print(f'Tracking URI: {MLFLOW_TRACKING}')
print(f'Total experimentos: {N_RUNS} seeds × {len(MODEL_NAMES)} modelos × {len(OPTIMIZER_NAMES)} optimizadores = {N_RUNS*len(MODEL_NAMES)*len(OPTIMIZER_NAMES)}')

## 2. Cargar datos

In [ ]:
X_train = np.load(os.path.join(ARRAYS_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(ARRAYS_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(ARRAYS_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(ARRAYS_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(ARRAYS_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(ARRAYS_DIR, 'y_test.npy'))
daylight_test = np.load(os.path.join(ARRAYS_DIR, 'daylight_test.npy'))
scaler_y = joblib.load(os.path.join(SCALERS_DIR, 'scaler_y.pkl'))

WINDOW_SIZE = X_train.shape[1]  # 24
N_FEATURES  = X_train.shape[2]  # 5

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Window: {WINDOW_SIZE}h  Features: {N_FEATURES}')

## 3. Espacio de búsqueda (compartido por todos los optimizadores)

In [ ]:
# Vector continuo para PSO / GA / ABC:
# [units_1, units_2, dropout, log10(lr), batch_idx]
LOWER = np.array([32,  16,  0.0, -4.0, 0.0])
UPPER = np.array([256, 128, 0.5, -2.0, 2.99])
DIM   = len(LOWER)

def decode(vec):
    """Convierte vector continuo a diccionario de hiperparámetros."""
    vec = np.clip(vec, LOWER, UPPER)
    batch_idx = int(np.floor(vec[4]))
    batch_idx = min(batch_idx, len(BATCH_CHOICES) - 1)
    return {
        'units_1':      int(round(vec[0])),
        'units_2':      int(round(vec[1])),
        'dropout_rate': float(vec[2]),
        'learning_rate': float(10 ** vec[3]),
        'batch_size':   BATCH_CHOICES[batch_idx],
    }

print('Espacio de búsqueda:')
print(f'  units_1:       [{int(LOWER[0])}, {int(UPPER[0])}]')
print(f'  units_2:       [{int(LOWER[1])}, {int(UPPER[1])}]')
print(f'  dropout_rate:  [{LOWER[2]:.1f}, {UPPER[2]:.1f}]')
print(f'  learning_rate: [1e-4, 1e-2] (log scale)')
print(f'  batch_size:    {BATCH_CHOICES}')

## 4. Constructores de modelos

In [ ]:
def build_mlp(params):
    model = Sequential([
        Flatten(input_shape=(WINDOW_SIZE, N_FEATURES)),
        Dense(params['units_1'], activation='relu'),
        Dropout(params['dropout_rate']),
        Dense(params['units_2'], activation='relu'),
        Dropout(params['dropout_rate']),
        Dense(1)
    ])
    model.compile(optimizer=Adam(params['learning_rate']), loss='mse')
    return model


def build_lstm(params):
    model = Sequential([
        LSTM(params['units_1'], return_sequences=True, input_shape=(WINDOW_SIZE, N_FEATURES)),
        Dropout(params['dropout_rate']),
        LSTM(params['units_2']),
        Dropout(params['dropout_rate']),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(params['learning_rate']), loss='mse')
    return model


def build_gru(params):
    model = Sequential([
        GRU(params['units_1'], return_sequences=True, input_shape=(WINDOW_SIZE, N_FEATURES)),
        Dropout(params['dropout_rate']),
        GRU(params['units_2']),
        Dropout(params['dropout_rate']),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(params['learning_rate']), loss='mse')
    return model


MODEL_BUILDERS = {'MLP': build_mlp, 'LSTM': build_lstm, 'GRU': build_gru}

## 5. Función objetivo y runner

In [ ]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


class BenchmarkRunner:
    """Encapsula el estado de un run: modelo, seed, historial anytime."""

    def __init__(self, model_type, seed):
        self.model_type   = model_type
        self.seed         = seed
        self.eval_count   = 0
        self.best_val_mae = float('inf')
        self.anytime      = []   # mejor MAE en cada evaluación

    def objective(self, params):
        """Entrena el modelo con `params` y devuelve val_MAE (a minimizar)."""
        try:
            tf.keras.backend.clear_session()
            set_seed(self.seed)

            model = MODEL_BUILDERS[self.model_type](params)
            cb = EarlyStopping(monitor='val_loss', patience=OPT_PATIENCE,
                               restore_best_weights=True)
            model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=OPT_EPOCHS,
                batch_size=params['batch_size'],
                callbacks=[cb],
                verbose=0
            )
            y_pred = model.predict(X_val, verbose=0)
            mae = mean_absolute_error(
                scaler_y.inverse_transform(y_val),
                scaler_y.inverse_transform(y_pred)
            )
        except Exception:
            mae = 9999.0

        self.eval_count += 1
        if mae < self.best_val_mae:
            self.best_val_mae = mae
        self.anytime.append(self.best_val_mae)

        # Log anytime al run activo de MLflow
        mlflow.log_metric('anytime_best_mae', self.best_val_mae, step=self.eval_count)
        return mae

    def objective_vec(self, vec):
        """Wrapper para PSO / GA / ABC (reciben vector numpy)."""
        return self.objective(decode(vec))

    def evaluate_final(self, params):
        """Entrena el modelo final con los mejores hiperparámetros y evalúa en test."""
        tf.keras.backend.clear_session()
        set_seed(self.seed)

        model = MODEL_BUILDERS[self.model_type](params)
        cb = EarlyStopping(monitor='val_loss', patience=FINAL_PATIENCE,
                           restore_best_weights=True)
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=FINAL_EPOCHS,
            batch_size=params['batch_size'],
            callbacks=[cb],
            verbose=0
        )
        y_pred = model.predict(X_test, verbose=0)
        y_true_inv = scaler_y.inverse_transform(y_test)
        y_pred_inv = scaler_y.inverse_transform(y_pred)

        mask = daylight_test
        return {
            'test_mae':     mean_absolute_error(y_true_inv, y_pred_inv),
            'test_rmse':    np.sqrt(mean_squared_error(y_true_inv, y_pred_inv)),
            'test_r2':      r2_score(y_true_inv, y_pred_inv),
            'daylight_mae': mean_absolute_error(y_true_inv[mask], y_pred_inv[mask]),
            'daylight_rmse':np.sqrt(mean_squared_error(y_true_inv[mask], y_pred_inv[mask])),
            'daylight_r2':  r2_score(y_true_inv[mask], y_pred_inv[mask]),
        }, model

## 6. Implementaciones de optimizadores

In [ ]:
# ── PSO ────────────────────────────────────────────────────────────────────────
def run_pso(objective_vec, seed, budget=100, n_particles=10,
            w=0.7, c1=1.5, c2=1.5):
    rng = np.random.default_rng(seed)
    n_iter = (budget // n_particles) - 1  # iteraciones tras inicialización

    pos = rng.uniform(LOWER, UPPER, (n_particles, DIM))
    vel = np.zeros((n_particles, DIM))
    pbest_pos = pos.copy()
    pbest_fit = np.array([objective_vec(p) for p in pos])

    gbest_idx = np.argmin(pbest_fit)
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_fit = pbest_fit[gbest_idx]

    for _ in range(n_iter):
        r1 = rng.random((n_particles, DIM))
        r2 = rng.random((n_particles, DIM))
        vel = w*vel + c1*r1*(pbest_pos - pos) + c2*r2*(gbest_pos - pos)
        pos = np.clip(pos + vel, LOWER, UPPER)

        for i in range(n_particles):
            f = objective_vec(pos[i])
            if f < pbest_fit[i]:
                pbest_fit[i] = f
                pbest_pos[i] = pos[i].copy()
                if f < gbest_fit:
                    gbest_fit = f
                    gbest_pos = pos[i].copy()

    return decode(gbest_pos), gbest_fit


# ── GA ─────────────────────────────────────────────────────────────────────────
def run_ga(objective_vec, seed, budget=100, pop_size=10,
           mutation_rate=0.2, elite_size=2):
    rng = np.random.default_rng(seed)
    n_gen = (budget // pop_size) - 1

    pop = rng.uniform(LOWER, UPPER, (pop_size, DIM))
    fit = np.array([objective_vec(ind) for ind in pop])

    def tournament(size=3):
        idx = rng.choice(pop_size, size, replace=False)
        return pop[idx[np.argmin(fit[idx])]].copy()

    best_fit = fit.min()
    best_ind = pop[np.argmin(fit)].copy()

    for _ in range(n_gen):
        new_pop = [pop[i].copy() for i in np.argsort(fit)[:elite_size]]
        while len(new_pop) < pop_size:
            p1, p2 = tournament(), tournament()
            alpha = rng.random(DIM)
            child = alpha*p1 + (1-alpha)*p2
            # mutación
            mask = rng.random(DIM) < mutation_rate
            child[mask] += rng.normal(0, 0.1*(UPPER-LOWER)[mask])
            new_pop.append(np.clip(child, LOWER, UPPER))

        pop = np.array(new_pop)
        fit = np.array([objective_vec(ind) for ind in pop])
        if fit.min() < best_fit:
            best_fit = fit.min()
            best_ind = pop[np.argmin(fit)].copy()

    return decode(best_ind), best_fit


# ── ABC ────────────────────────────────────────────────────────────────────────
def run_abc(objective_vec, seed, budget=100, n_bees=10, limit=15):
    rng = np.random.default_rng(seed)
    n_employed = n_bees // 2
    evals = 0

    sources = rng.uniform(LOWER, UPPER, (n_employed, DIM))
    fitness = np.array([objective_vec(s) for s in sources])
    trials  = np.zeros(n_employed)
    evals  += n_employed

    best_fit = fitness.min()
    best_src = sources[np.argmin(fitness)].copy()

    while evals < budget:
        # Fase empleadas
        for i in range(n_employed):
            if evals >= budget: break
            k = rng.choice([j for j in range(n_employed) if j != i])
            phi = rng.uniform(-1, 1, DIM)
            cand = np.clip(sources[i] + phi*(sources[i]-sources[k]), LOWER, UPPER)
            f = objective_vec(cand); evals += 1
            if f < fitness[i]:
                sources[i] = cand; fitness[i] = f; trials[i] = 0
                if f < best_fit: best_fit = f; best_src = cand.copy()
            else:
                trials[i] += 1

        # Fase observadoras
        probs = (1/(fitness+1e-9)); probs /= probs.sum()
        for _ in range(n_employed):
            if evals >= budget: break
            i = rng.choice(n_employed, p=probs)
            k = rng.choice([j for j in range(n_employed) if j != i])
            phi = rng.uniform(-1, 1, DIM)
            cand = np.clip(sources[i] + phi*(sources[i]-sources[k]), LOWER, UPPER)
            f = objective_vec(cand); evals += 1
            if f < fitness[i]:
                sources[i] = cand; fitness[i] = f; trials[i] = 0
                if f < best_fit: best_fit = f; best_src = cand.copy()
            else:
                trials[i] += 1

        # Fase exploradoras
        for i in range(n_employed):
            if trials[i] > limit and evals < budget:
                sources[i] = rng.uniform(LOWER, UPPER)
                fitness[i] = objective_vec(sources[i]); evals += 1
                trials[i] = 0
                if fitness[i] < best_fit: best_fit = fitness[i]; best_src = sources[i].copy()

    return decode(best_src), best_fit


# ── TPE (Optuna) ───────────────────────────────────────────────────────────────
def run_tpe(runner, seed, budget=100):
    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction='minimize', sampler=sampler)

    def optuna_objective(trial):
        params = {
            'units_1':      trial.suggest_int('units_1', 32, 256),
            'units_2':      trial.suggest_int('units_2', 16, 128),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.0, 0.5),
            'learning_rate':trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
            'batch_size':   trial.suggest_categorical('batch_size', BATCH_CHOICES),
        }
        return runner.objective(params)

    study.optimize(optuna_objective, n_trials=budget)
    return study.best_params, study.best_value


OPTIMIZERS = {
    'PSO': lambda runner, seed: run_pso(runner.objective_vec, seed, BUDGET),
    'GA':  lambda runner, seed: run_ga(runner.objective_vec, seed, BUDGET),
    'ABC': lambda runner, seed: run_abc(runner.objective_vec, seed, BUDGET),
    'TPE': lambda runner, seed: run_tpe(runner, seed, BUDGET),
}

## 7. Loop principal del benchmark

**Tiempo estimado:** ~2–3 días en RTX 4070.  
Monitorear en `http://localhost:5000` mientras corre.

In [ ]:
checkpoint_path = os.path.join(RESULTS_DIR, 'benchmark_results.csv')
completed = set()

if os.path.exists(checkpoint_path):
    try:
        prev = pd.read_csv(checkpoint_path)
        if prev.empty:
            raise ValueError('CSV vacío')
        completed = set(zip(prev['model'], prev['optimizer'], prev['seed']))
        print(f'Checkpoint encontrado: {len(completed)} runs completados.')
    except Exception as e:
        import shutil
        bak = checkpoint_path + '.bak'
        shutil.copy(checkpoint_path, bak)
        print(f'⚠ Checkpoint corrupto ({e}) — backup en {bak}, iniciando desde cero.')
        prev = pd.DataFrame()
        completed = set()
else:
    prev = pd.DataFrame()
    print('Iniciando benchmark desde cero.')

all_rows = prev.to_dict('records') if not prev.empty else []

In [ ]:
total     = len(MODEL_NAMES) * len(OPTIMIZER_NAMES) * N_RUNS
run_times = []

pbar = tqdm(total=total, initial=len(completed),
            desc='Benchmark', unit='run',
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')

for seed in range(N_RUNS):
    for model_type in MODEL_NAMES:
        for opt_name in OPTIMIZER_NAMES:

            if (model_type, opt_name, seed) in completed:
                continue

            run_name = f'{model_type}_{opt_name}_s{seed:02d}'
            pbar.set_postfix_str(run_name, refresh=True)

            with mlflow.start_run(run_name=run_name):
                mlflow.set_tags({'model': model_type, 'optimizer': opt_name, 'seed': seed})
                mlflow.log_params({
                    'model': model_type, 'optimizer': opt_name, 'seed': seed,
                    'budget': BUDGET, 'window': WINDOW_SIZE, 'n_features': N_FEATURES,
                })

                runner = BenchmarkRunner(model_type, seed)
                t0 = time.time()

                # ── Optimización ──────────────────────────────────────────────
                try:
                    best_params, best_val_mae = OPTIMIZERS[opt_name](runner, seed)
                except Exception as e:
                    tqdm.write(f'  ✗ OPT FAIL {run_name}: {e}')
                    mlflow.set_tag('status', 'failed_opt')
                    pbar.update(1)
                    continue
                opt_time = time.time() - t0

                # ── Evaluación final en test ───────────────────────────────────
                try:
                    test_metrics, final_model = runner.evaluate_final(best_params)
                except Exception as e:
                    tqdm.write(f'  ✗ EVAL FAIL {run_name}: {e}')
                    mlflow.set_tag('status', 'failed_eval')
                    pbar.update(1)
                    continue
                run_time = time.time() - t0
                run_times.append(run_time)

                # ── Log a MLflow ───────────────────────────────────────────────
                mlflow.log_params({f'hp_{k}': v for k, v in best_params.items()})
                mlflow.log_metrics({
                    'best_val_mae': best_val_mae, 'opt_time_s': opt_time,
                    'n_evals': runner.eval_count, **test_metrics,
                })
                model_path = os.path.join(MODELS_DIR, f'{run_name}.keras')
                final_model.save(model_path)
                mlflow.log_artifact(model_path)
                mlflow.set_tag('status', 'completed')

            # ── Checkpoint atómico (write-then-rename) ────────────────────────
            row = {
                'model': model_type, 'optimizer': opt_name, 'seed': seed,
                'best_val_mae': best_val_mae, 'opt_time_s': opt_time,
                'n_evals': runner.eval_count,
                **test_metrics,
                **{f'hp_{k}': v for k, v in best_params.items()},
            }
            all_rows.append(row)
            tmp_path = checkpoint_path + '.tmp'
            pd.DataFrame(all_rows).to_csv(tmp_path, index=False)
            os.replace(tmp_path, checkpoint_path)   # atómico en POSIX

            # ── Progreso ──────────────────────────────────────────────────────
            runs_left = total - (len(completed) + len(run_times))
            eta_str = '?'
            if run_times:
                avg_s  = np.mean(run_times[-20:])
                eta_dt = datetime.now() + timedelta(seconds=avg_s * runs_left)
                eta_str = eta_dt.strftime('%Y-%m-%d %H:%M')

            tqdm.write(
                f'  {run_name}  val={best_val_mae:.2f}  '
                f'test={test_metrics["test_mae"]:.2f}  '
                f'dl={test_metrics["daylight_mae"]:.2f}  '
                f'{run_time/60:.1f}min  ETA {eta_str}'
            )
            pbar.update(1)

pbar.close()
print('\n✓ Benchmark completo.')

## 8. Resumen de resultados

In [ ]:
results = pd.read_csv(checkpoint_path)
print(f'Total runs completados: {len(results)}')
results.head()

In [ ]:
# Tabla resumen: media ± std por (modelo, optimizador)
summary = results.groupby(['model', 'optimizer']).agg(
    test_mae_mean   =('test_mae',     'mean'),
    test_mae_std    =('test_mae',     'std'),
    test_rmse_mean  =('test_rmse',    'mean'),
    test_rmse_std   =('test_rmse',    'std'),
    test_r2_mean    =('test_r2',      'mean'),
    dl_mae_mean     =('daylight_mae', 'mean'),
    dl_mae_std      =('daylight_mae', 'std'),
    dl_r2_mean      =('daylight_r2',  'mean'),
    opt_time_min    =('opt_time_s',   lambda x: x.mean()/60),
    n_runs          =('seed',         'count'),
).round(3)

summary.to_csv(os.path.join(RESULTS_DIR, 'benchmark_summary.csv'))
summary

In [ ]:
print('\nPara ver todos los runs en detalle:')
print(f'  mlflow ui --port 5000 --backend-store-uri file://{MLFLOW_TRACKING}')
print(f'\nResultados guardados en: {RESULTS_DIR}')
print('  benchmark_results.csv  — un run por fila')
print('  benchmark_summary.csv  — media ± std por (modelo, optimizador)')
print('\n✓ Continuar con 04_analysis.ipynb para pruebas estadísticas.')